<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0">

# Delta Hedging

This part of the tutorial is concerned with implementation of delta hedging strategies for European call and put options in the Black-Scholes-Merton (1973) model.

In [ ]:
!git clone https://github.com/tpq-classes/computational_finance_dx.git
import sys
sys.path.append('computational_finance_dx')


In [ ]:
from dx import *
from pylab import plt
plt.style.use('seaborn-v0_8')
%matplotlib inline

## Risk Factor

This sub-section models the single risk factor. We start with the definition of the risk-neutral discounting object.

In [ ]:
# constant short rate
r = constant_short_rate('r', 0.06)

The simulation model of choice is the **geometric Brownian motion**.

In [ ]:
# market environment
me_gbm = market_environment('gbm', dt.datetime(2020, 1, 1))

Assumptions for the `geometric_brownian_motion` object.

In [ ]:
# geometric Brownian motion
me_gbm.add_constant('initial_value', 36.)
me_gbm.add_constant('volatility', 0.2) 
me_gbm.add_constant('currency', 'EUR')
me_gbm.add_constant('paths', 10000)
me_gbm.add_constant('frequency', 'W')
me_gbm.add_curve('discount_curve', r)
me_gbm.add_constant('starting_date', dt.datetime(2020, 1, 1))
me_gbm.add_constant('final_date', dt.datetime(2020, 12, 31))

In [ ]:
gbm = geometric_brownian_motion('gbm', me_gbm)

## Derivatives

In this sub-section, we model the single derivatives instruments.

### European Call option

The first derivative instrument is a **European call option**.

In [ ]:
me_call = market_environment('call', dt.datetime(2020, 1, 1))
me_call.add_constant('maturity', dt.datetime(2020, 12, 31))
me_call.add_constant('strike', 36)
me_call.add_constant('currency', 'EUR')
me_call.add_environment(me_gbm)

In [ ]:
payoff_call = 'np.maximum(maturity_value - strike, 0)'

In [ ]:
eur_call = valuation_mcs_european_single(
                    'eur_call',
                    mar_env=me_call,
                    underlying=gbm,
                    payoff_func=payoff_call)

In [ ]:
eur_call.present_value(fixed_seed=False)

In [ ]:
eur_call.delta()

### European Put Option

The second derivative instrument is a **European put option**.

In [ ]:
me_put = market_environment('put', dt.datetime(2020, 1, 1))
me_put.add_constant('maturity', dt.datetime(2020, 12, 31))
me_put.add_constant('strike', 40.)
me_put.add_constant('currency', 'EUR')
me_put.add_environment(me_gbm)

In [ ]:
payoff_put = 'np.maximum(strike - maturity_value, 0)'

In [ ]:
eur_put = valuation_mcs_european_single(
                    'eur_put',
                    mar_env=me_put,
                    underlying=gbm,
                    payoff_func=payoff_put)

In [ ]:
eur_put.present_value()

In [ ]:
eur_put.delta()

## Delta Hedging

### European Call

First, the European call option is hedged.

In [ ]:
me_gbm.add_constant('initial_value', 36.)
me_gbm.add_constant('frequency', 'W')
me_gbm.add_constant('paths', 10000)
me_gbm.pricing_date = dt.datetime(2020, 1, 1)
gbm = geometric_brownian_motion('gbm', me_gbm)

A certain path is chosen to simulate the delta hedging.

In [ ]:
no = 26

In [ ]:
data = pd.DataFrame(gbm.get_instrument_values()[:, no],
                    index=pd.DatetimeIndex(gbm.time_grid),
                    columns=['price'])

In [ ]:
data.plot(figsize=(10, 6));

In [ ]:
dti = (data.index[-1] - data.index[0]).days / 365 / len(data)
dti

In [ ]:
data['option'] = 0.0
data['delta'] = 0.0
data['stock'] = 0.0
data['bond'] = 0.0
data['hedge'] = 0.0
data['error'] = 0.0

The delta hedging code. It stores all relevant statistics in a `DataFrame` object.

In [ ]:
for i, row in enumerate(list(data.iterrows())[:-1]):
    me_gbm.pricing_date = row[0].to_pydatetime()
    price = row[1]['price']
    me_gbm.add_constant('initial_value', price)
    me_call.add_environment(me_gbm)
    gbm = geometric_brownian_motion('gbm', me_gbm)
    eur_call = valuation_mcs_european_single(
                        'eur_call',
                        mar_env=me_call,
                        underlying=gbm,
                        payoff_func=payoff_call)
    value = eur_call.present_value()
    data.loc[row[0], 'option'] = value
    delta = eur_call.delta()
    data.loc[row[0], 'delta'] = delta
    data.loc[row[0], 'stock'] = delta * price
    data.loc[row[0], 'bond'] = value - delta * price
    if i == 0:
        data.loc[row[0], 'hedge'] = (data.loc[row[0], 'delta'] * price +
                                     data.loc[row[0], 'bond'])
    else:
        data.loc[row[0], 'hedge'] = (data.loc[prev, 'delta'] * price +
                                     data.loc[prev, 'bond'] * math.exp(r.short_rate * dti)) 
        data.loc[row[0], 'error'] = data.loc[row[0], 'hedge'] - data.loc[row[0], 'option']
    prev = row[0]
data.iloc[-1, 1] = max(data['price'].iloc[-1] - eur_call.strike, 0)
data.iloc[-1, 5] = (data.loc[prev, 'delta'] * data['price'].iloc[-1] +
                    data.loc[prev, 'bond'] * math.exp(r.short_rate * dti))

Underlying price vs. option value.

In [ ]:
data[['price', 'option']].plot(figsize=(10, 6), secondary_y='option');

Option value vs. delta.

In [ ]:
data[['option', 'delta']].plot(figsize=(10, 6), secondary_y='delta');

The hedge portfolio value vs. the option value. Not perfect but close.

In [ ]:
data[['option', 'hedge']].plot(figsize=(10, 6));

Having a look at the error values (P&L over time).

In [ ]:
data['error'].hist(figsize=(10, 6), bins=35);

In [ ]:
data['error'].mean()

In [ ]:
data['error'].sum()

In [ ]:
data['error'].cumsum().plot(figsize=(10, 6));

### European Put

Second, the European put option is hedged.

In [ ]:
me_gbm.add_constant('initial_value', 36.)
me_gbm.add_constant('frequency', 'B')
me_gbm.pricing_date = dt.datetime(2020, 1, 1)
gbm = geometric_brownian_motion('gbm', me_gbm)

A certain path is chosen to simulate the delta hedging.

In [ ]:
no = 100

In [ ]:
data = pd.DataFrame(gbm.get_instrument_values()[:, no],
                    index=pd.DatetimeIndex(gbm.time_grid),
                    columns=['price'])

In [ ]:
dti = (data.index[-1] - data.index[0]).days / 365 / len(data)
dti

In [ ]:
data['option'] = 0.0
data['delta'] = 0.0
data['stock'] = 0.0
data['bond'] = 0.0
data['hedge'] = 0.0
data['error'] = 0.0

The delta hedging code. It stores all relevant statistics in a `DataFrame` object.

In [ ]:
%%time
for i, row in enumerate(list(data.iterrows())[:-1]):
    me_gbm.pricing_date = row[0].to_pydatetime()
    price = row[1]['price']
    me_gbm.add_constant('initial_value', price)
    me_put.add_environment(me_gbm)
    gbm = geometric_brownian_motion('gbm', me_gbm)
    eur_put = valuation_mcs_european_single(
                        'eur_put',
                        mar_env=me_put,
                        underlying=gbm,
                        payoff_func=payoff_put)
    value = eur_put.present_value()
    data.loc[row[0], 'option'] = value
    delta = eur_put.delta()
    data.loc[row[0], 'delta'] = delta
    data.loc[row[0], 'stock'] = delta * price
    data.loc[row[0], 'bond'] = value - delta * price
    if i == 0:
        data.loc[row[0], 'hedge'] = (data.loc[row[0], 'delta'] * price +
                                     data.loc[row[0], 'bond'])
    else:
        data.loc[row[0], 'hedge'] = (data.loc[prev, 'delta'] * price +
                                     data.loc[prev, 'bond'] * math.exp(r.short_rate * dti))
        data.loc[row[0], 'error'] = data.loc[row[0], 'hedge'] - data.loc[row[0], 'option']
    prev = row[0]
data.iloc[-1, 1] = max(eur_put.strike - data['price'].iloc[-1], 0)
data.iloc[-1, 5] = (data.loc[prev, 'delta'] * data['price'].iloc[-1] +
                    data.loc[prev, 'bond'] * math.exp(r.short_rate * dti))

Underlying price vs. option value.

In [ ]:
data[['price', 'option']].plot(figsize=(10, 6), secondary_y='option');

Option value vs. delta.

In [ ]:
data[['option', 'delta']].plot(figsize=(10, 6), secondary_y='delta');

The hedge portfolio value vs. the option value. Not perfect but close.

In [ ]:
data[['option', 'hedge']].plot(figsize=(10, 6));

Having a look at the error values (P&L over time).

In [ ]:
data['error'].hist(figsize=(10, 6), bins=35);

In [ ]:
data['error'].mean()

In [ ]:
data['error'].sum()

In [ ]:
data['error'].cumsum().plot(figsize=(10, 6));

**Copyright, License & Disclaimer**

© Dr. Yves J. Hilpisch | The Python Quants GmbH

DX Analytics (the "dx library" or "dx package") is licensed under the GNU Affero General
Public License version 3 or later (see http://www.gnu.org/licenses/).

DX Analytics comes with no representations or warranties, to the extent
permitted by applicable law.

http://tpq.io | [dx@tpq.io](mailto:team@tpq.io) |
http://twitter.com/dyjh

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

**Quant Platform** | http://pqp.io

**Python for Finance Training** | http://training.tpq.io

**Certificate in Computational Finance** | http://compfinance.tpq.io

**Derivatives Analytics with Python (Wiley Finance)** |
http://dawp.tpq.io

**Python for Finance (2nd ed., O'Reilly)** |
http://py4fi.tpq.io